In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
df = pd.read_csv(r"C:\Users\Lenovo\Downloads\housing_prices.csv")

X = df.drop(columns=['PropertyID', 'Price'])
y = df['Price']

In [3]:
num_cols = ['SquareFeet', 'Bedrooms', 'Bathrooms', 'AgeYears', 'GarageSpaces']
cat_cols = ['Neighborhood']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

In [6]:
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(random_state=42))
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(random_state=42))
])

In [7]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("--- Cross-Validation Model Comparison (R² Score) ---")
for name, pipeline in [("Ridge Regression", ridge_pipeline),
                       ("Random Forest Regressor", rf_pipeline),
                       ("Gradient Boosting Regressor", gb_pipeline)]:
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='r2')
    print(f"{name} CV R²: {scores.mean():.4f} (+/- {scores.std():.4f})")

--- Cross-Validation Model Comparison (R² Score) ---
Ridge Regression CV R²: 0.9268 (+/- 0.0146)
Random Forest Regressor CV R²: 0.9595 (+/- 0.0080)
Gradient Boosting Regressor CV R²: 0.9647 (+/- 0.0106)


In [8]:
param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__max_depth': [3, 5]
}

grid_search = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\n--- Hyperparameter Tuning ---")
print("Best Parameters:", grid_search.best_params_)
print(f"Best CV R² Score: {grid_search.best_score_:.4f}")


--- Hyperparameter Tuning ---
Best Parameters: {'regressor__learning_rate': 0.05, 'regressor__max_depth': 5, 'regressor__n_estimators': 200}
Best CV R² Score: 0.9666


In [9]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("\n--- Test Set Evaluation ---")
print(f"R² Score: {r2:.4f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")


--- Test Set Evaluation ---
R² Score: 0.9699
Root Mean Squared Error (RMSE): $59,792.76
Mean Absolute Error (MAE): $38,147.04


In [10]:
new_house = pd.DataFrame({
    'SquareFeet': [2400],
    'Bedrooms': [3],
    'Bathrooms': [2.5],
    'AgeYears': [5],
    'GarageSpaces': [2],
    'Neighborhood': ['Waterfront']  # Options: 'Downtown', 'Suburbs', 'Rural', 'Waterfront'
})

In [11]:
predicted_price = best_model.predict(new_house)

print(f"Predicted House Price: ${predicted_price[0]:,.2f}")

Predicted House Price: $1,193,433.28
